In [2]:
from pathlib import Path
import pandas as pd
import json
import re

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

# Frozen KGL
kgl = pd.read_csv(
    PROJECT_ROOT / "data/purification/self_assertion_kgl_FINAL.csv"
)

# Cleaned focal sentences
focal = pd.read_csv(
    PROJECT_ROOT / "data/preprocessed/focal_sentences_clean.csv"
)

# Extracted matched sentences from Notebook 06
contexts = pd.read_csv(
    PROJECT_ROOT / "data/contexts/focal_self_assertion_contexts.csv"
)

print("KGL terms:", len(kgl))
print("Focal sentences:", len(focal))
print("Matched sentences:", len(contexts))

KGL terms: 62
Focal sentences: 18882
Matched sentences: 1230


In [3]:
# Count words in each cleaned sentence
focal["word_count"] = focal["sentence_text_lower"].apply(
    lambda text: len(re.findall(r"\b\w+\b", text))
)

# Aggregate word counts to speech level
speech_lengths = (
    focal.groupby(["iso3", "year", "filename"], as_index=False)
    ["word_count"].sum()
)

assert len(speech_lengths) == 180
assert (speech_lengths["word_count"] > 0).all()

print("Speeches:", len(speech_lengths))
print("Median speech length:", speech_lengths["word_count"].median())

Speeches: 180
Median speech length: 2020.5


In [4]:
# Create dictionary: term -> frozen weight
weights = dict(zip(
    kgl["term"].str.lower().str.strip(),
    kgl["weight"]
))

# Convert saved JSON strings back into lists
contexts["matched_terms"] = contexts["matched_terms"].apply(
    json.loads
)

# Calculate weighted matches per sentence
contexts["weighted_matches"] = contexts["matched_terms"].apply(
    lambda terms: sum(weights[t] for t in terms)
)

# Aggregate to speech level
speech_matches = (
    contexts.groupby(["iso3", "year", "filename"], as_index=False)
    .agg(
        n_matches=("n_matches", "sum"),
        weighted_matches=("weighted_matches", "sum")
    )
)

assert speech_matches["n_matches"].sum() == 1309

print("Total raw matches:", speech_matches["n_matches"].sum())
print("Total weighted matches:", speech_matches["weighted_matches"].sum())

Total raw matches: 1309
Total weighted matches: 1046.7232205867767


In [5]:
# Merge speech lengths and matched frequencies
scores = speech_lengths.merge(
    speech_matches,
    on=["iso3", "year", "filename"],
    how="left",
    validate="one_to_one"
)

# Speeches without KGL matches receive zero
scores[["n_matches", "weighted_matches"]] = (
    scores[["n_matches", "weighted_matches"]].fillna(0)
)

scores["n_matches"] = scores["n_matches"].astype(int)

# Main Self-Assertion score
scores["sa_per_1000"] = (
    scores["weighted_matches"]
    / scores["word_count"]
    * 1000
)

# Final checks
assert len(scores) == 180
assert scores["n_matches"].sum() == 1309
assert scores["sa_per_1000"].notna().all()

# Save results
output = PROJECT_ROOT / "data/results/focal_self_assertion_scores.csv"
output.parent.mkdir(parents=True, exist_ok=True)

scores.to_csv(output, index=False)

print("Saved:", output)
print("\nSelf-Assertion score distribution:")
print(scores["sa_per_1000"].describe())

display(scores.head(10))

Saved: /Users/catherinexie/Documents/kgl-mockproject/data/results/focal_self_assertion_scores.csv

Self-Assertion score distribution:
count    180.000000
mean       2.600099
std        1.464499
min        0.000000
25%        1.493911
50%        2.484605
75%        3.546500
max        7.337937
Name: sa_per_1000, dtype: float64


,iso3,year,filename,word_count,n_matches,weighted_matches,sa_per_1000
0,CHL,2016,CHL_71_2016.txt,1312,7,4.862996,3.706552
1,CHL,2017,CHL_72_2017.txt,2568,8,6.582251,2.563182
2,CHL,2018,CHL_73_2018.txt,4397,2,1.453595,0.330588
3,CHL,2019,CHL_74_2019.txt,2752,5,3.735143,1.357247
4,CHL,2020,CHL_75_2020.txt,3104,4,3.080814,0.992530
5,CHL,2021,CHL_76_2021.txt,2982,5,4.112977,1.379268
6,CHL,2022,CHL_77_2022.txt,2515,11,8.158109,3.243781
7,CHL,2023,CHL_78_2023.txt,2848,17,13.499870,4.740123
8,CHL,2024,CHL_79_2024.txt,1937,15,11.644528,6.011630
9,CHL,2025,CHL_80_2025.txt,2997,8,6.815670,2.274164
